# Simulation Behavior — per experiment

Six diagnostic plots over the full run of every simulation experiment (latest trajectory per experiment):

1. Online vs. offline users over time
2. Active requests over time, per request type
3. Total CPU usage: users (online) vs. server
4. Stored (ready) subtask partitions: all users (online+offline) vs. online users only
5. Required vs. currently executing subtasks (server + users combined)
6. Cumulative initiated requests, per request type

Figures are saved as `sim{idx}_<plot>.pdf`.

In [ ]:
import os
import sys
root = os.path.abspath("..")
sys.path.append(root)

import io
import pickle
import shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import scienceplots

from config import Config
from simulation.impl.trajectory_with_metrics import TrajectoryWithMetrics

if shutil.which("latex"):
    plt.style.use(["science", "ieee"])
else:
    plt.style.use(["science", "ieee", "no-latex"])

config = Config()
figures_dir = os.path.join(config.resource_dir, "figures")
os.makedirs(figures_dir, exist_ok=True)

# LNCS single-column width: ~8.5 cm (same as available_resources.ipynb)
FIG_W = 8.5 / 2.54
FIG_H = 5.0 / 2.54

# Font sizes: same as available_resources.ipynb
_FONT_SCALE  = FIG_W / 4.0
LABEL_FONTSIZE = 12 * _FONT_SCALE
XTICK_FONTSIZE = 14 * _FONT_SCALE
YTICK_FONTSIZE = 10 * _FONT_SCALE


def style_axes(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel, fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE * 0.85)
    ax.tick_params(axis="x", labelsize=XTICK_FONTSIZE)
    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)
    ax.grid(True, linestyle="--", alpha=0.4)


_theme_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

def smooth(values, window=1000):
    """Centered rolling mean to de-noise highly stochastic per-timestep series."""
    return pd.Series(values).rolling(window, center=True, min_periods=1).mean().to_numpy()

REQUEST_TYPE_NAMES = {0: "Light 10", 1: "Heavy 10", 2: "Light 5", 3: "Heavy 5"}

LONG_DASHES = {
    "-": "-",
    "--": (0, (6, 3)),
    "-.": (0, (6, 2, 1.5, 2)),
    ":": (0, (1.5, 2)),
}

# Marker per request type, independent of color/linestyle — in plot 4 color
# distinguishes type and linestyle distinguishes online-vs-all, so two types that
# happen to land on a red/green pair (the classic colorblind confusion) would
# otherwise be indistinguishable without this.
REQUEST_TYPE_MARKERS = {0: "o", 1: "s", 2: "^", 3: "D"}

## Discover experiments

In [ ]:
import re
import json
from datetime import datetime


class CPU_Unpickler(pickle.Unpickler):
    """Experiments ran on a CUDA device; unpickling their tensors on a CPU-only
    machine needs them remapped at load time."""
    def find_class(self, module, name):
        if module == "torch.storage" and name == "_load_from_bytes":
            return lambda b: torch.load(io.BytesIO(b), map_location="cpu")
        return super().find_class(module, name)


TRAJECTORY_DIR = "path/to/trajectories"
EXPERIMENT_DIR = os.path.abspath("../experiments")
CLIENT_CPU_BASE = 200.0


def load_records():
    """Latest trajectory per experiment index, paired with its experiment config."""
    latest = {}
    for filename in os.listdir(TRAJECTORY_DIR):
        match = re.fullmatch(r"trajectory_exp_simulation_(.+)_(\d+)_(.+)\.pkl", filename)
        if not match:
            continue
        device, idx, date_str = match.groups()
        idx, date = int(idx), datetime.fromisoformat(date_str)
        if idx not in latest or date > latest[idx][0]:
            latest[idx] = (date, device, os.path.join(TRAJECTORY_DIR, filename))

    records = []
    for idx in sorted(latest):
        _, device, path = latest[idx]
        exp_path = os.path.join(EXPERIMENT_DIR, f"exp_simulation_{device}_{idx}.json")
        if not os.path.exists(exp_path):
            continue
        with open(exp_path) as f:
            records.append((idx, path, json.load(f)))
    return records


def describe(idx, exp):
    server_mult = int(round(exp["available_resource_params"]["server"]["cpu"]["max"] / CLIENT_CPU_BASE))
    client_mult = int(round(exp["available_resource_params"]["client"]["cpu"]["max"] / CLIENT_CPU_BASE))
    policy = exp["server_policy"]
    if policy == "smart" and float(exp.get("server_policy_smart_priority_scale", 1.0)) > 1.0:
        policy = "smartx"
    return (f"Experiment {idx}: {exp['n_clients']} users, {client_mult}x client / "
            f"{server_mult}x server capacity, {policy} policy")


records = load_records()
print(f"Found {len(records)} experiments")

## 1. Online vs. offline users

In [ ]:
def plot_online_offline_users(trajectory, timesteps, idx):
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.plot(timesteps, (trajectory.n_online_clients), color=_theme_colors[3],
            linestyle=LONG_DASHES["-"], linewidth=1.2, label="Online users")
    ax.plot(timesteps, (trajectory.n_offline_clients), color=_theme_colors[1],
            linestyle=LONG_DASHES[":"], linewidth=1.2, label="Offline users")
    style_axes(ax, "Time (s)", "Number of users")
    ax.legend(fontsize=8 * _FONT_SCALE)
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f"sim{idx}_online_offline_users.pdf"), bbox_inches="tight")
    plt.show()

## 2. Active requests over time, per request type

In [ ]:
# _REQUEST_LINESTYLES = [LONG_DASHES["-"], LONG_DASHES["--"], LONG_DASHES["-."], LONG_DASHES[":"]]
_REQUEST_LINESTYLES = [LONG_DASHES["--"], LONG_DASHES["-."], LONG_DASHES["-"], LONG_DASHES[":"]]


def plot_active_requests(trajectory, timesteps, idx):
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    for t in [2, 0, 3, 1]:  # plot in order of increasing request size (light first, then heavy)
        ax.plot(timesteps, (trajectory.active_queries[t]),
                    color=_theme_colors[t % len(_theme_colors)],
                    linestyle=_REQUEST_LINESTYLES[t % len(_REQUEST_LINESTYLES)],
                    linewidth=1.2,
                    alpha=0.1)
        ax.plot(timesteps, smooth(trajectory.active_queries[t]),
                color=_theme_colors[t % len(_theme_colors)],
                linestyle=_REQUEST_LINESTYLES[t % len(_REQUEST_LINESTYLES)],
                linewidth=1.2, label=REQUEST_TYPE_NAMES[t])
    style_axes(ax, "Time (s)", "Active requests (smoothed)")
    ax.legend(fontsize=8 * _FONT_SCALE)
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f"sim{idx}_active_requests.pdf"), bbox_inches="tight")
    plt.show()

## 3. Total CPU usage: users (online) vs. server

In [ ]:
def plot_cpu_usage(trajectory, timesteps, idx):
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

    ax.plot(timesteps, (trajectory.server_cpu_usage), color=_theme_colors[3],
            linestyle=LONG_DASHES["-"], linewidth=1.2, alpha=0.1)
    ax.plot(timesteps, (trajectory.online_client_total_cpu_usage), color=_theme_colors[2],
            linestyle=LONG_DASHES["--"], linewidth=1.2, alpha=0.1)

    ax.plot(timesteps, smooth(trajectory.server_cpu_usage), color=_theme_colors[3],
            linestyle=LONG_DASHES["-"], linewidth=1.2, label="Server")
    ax.plot(timesteps, smooth(trajectory.online_client_total_cpu_usage), color=_theme_colors[2],
            linestyle=LONG_DASHES["--"], linewidth=1.2, label="Users (online)")
    style_axes(ax, "Time (s)", "Total CPU use (\%) (smoothed)")
    ax.legend(fontsize=8 * _FONT_SCALE)
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f"sim{idx}_cpu_usage.pdf"), bbox_inches="tight")
    plt.show()

## 4. Stored (ready) subtask partitions: all users vs. online users only

In [ ]:
_READY_LINESTYLES = {"all": LONG_DASHES["-"], "online": LONG_DASHES["--"]}


def plot_stored_partitions(trajectory, timesteps, idx):
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    _online_curves = []
    for t in [2, 0, 3, 1]:  # plot in order of increasing request size (light first, then heavy)
        online_ready_t = smooth(np.array(trajectory.online_client_total_ready_x_subtasks[t]))
        all_ready_t = online_ready_t + smooth(np.array(trajectory.offline_client_total_ready_x_subtasks[t]))
        color = _theme_colors[t % len(_theme_colors)]
        _online_curves.append(online_ready_t)

        ax.plot(timesteps, all_ready_t, color=color, linestyle=_READY_LINESTYLES["all"],
                marker=REQUEST_TYPE_MARKERS[t], markersize=3.5, markevery=0.07, markeredgewidth=0.6,
                linewidth=1.2, label=f"{REQUEST_TYPE_NAMES[t]}, all users")
        ax.plot(timesteps, online_ready_t, color=color, linestyle=_READY_LINESTYLES["online"],
                marker=REQUEST_TYPE_MARKERS[t], markersize=3.5, markevery=0.07, markeredgewidth=0.6,
                linewidth=1.2, label=f"{REQUEST_TYPE_NAMES[t]}, online only")

    # Shade the band the "online only" curves live in, so the online-vs-all split reads
    # at a glance without having to parse 8 individual lines.
    _online_band_top = np.concatenate(_online_curves).max() * 1.15
    ax.axhspan(0, _online_band_top, color="#2ca02c", alpha=0.15, zorder=0, linewidth=0)
    ax.text(timesteps[-1] * 0.98, _online_band_top, "Online",
            fontsize=7 * _FONT_SCALE, color="#2ca02c", style="italic", ha="right", va="bottom")

    style_axes(ax, "Time (s)", "Stored subtasks (smoothed)")
    ax.legend(fontsize=5.5 * _FONT_SCALE, ncol=2, columnspacing=0.8, handletextpad=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f"sim{idx}_stored_partitions.pdf"), bbox_inches="tight")
    plt.show()

## 5. Required vs. currently executing subtasks (server + users)

In [ ]:
def plot_required_vs_executing(trajectory, timesteps, idx):
    # Required: total y-subtasks demanded by currently active requests, minus those
    # already finished (DONE) for those requests — i.e. remaining demand, not raw total.
    required_raw = np.array(trajectory.required_total_y_subtasks).sum(axis=0)
    done = (np.array(trajectory.server_total_done_y_subtasks)
            + np.array(trajectory.online_client_total_done_y_subtasks)).sum(axis=0)
    required = np.clip(required_raw - done, 0, None)

    # In progress: any non-terminal state (downloading, executing, uploading). The server
    # has no download/upload phase for y-subtasks (it already holds the models), so only
    # its executing count applies there.
    in_progress = (
        np.array(trajectory.server_total_executing_y_subtasks).sum(axis=0)
        + np.array(trajectory.online_client_total_downloading_y_subtasks).sum(axis=0)
        + np.array(trajectory.online_client_total_executing_y_subtasks).sum(axis=0)
        + np.array(trajectory.online_client_total_uploading_y_subtasks).sum(axis=0)
    )

    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.plot(timesteps, (required), color=_theme_colors[1],
            linestyle=LONG_DASHES["-"], linewidth=1.2, alpha=0.1)
    ax.plot(timesteps, (in_progress), color=_theme_colors[2],
            linestyle=LONG_DASHES["--"], linewidth=1.2, alpha=0.1)
    ax.plot(timesteps, smooth(required), color=_theme_colors[1],
            linestyle=LONG_DASHES["-"], linewidth=1.2, label="Required")
    ax.plot(timesteps, smooth(in_progress), color=_theme_colors[2],
            linestyle=LONG_DASHES["--"], linewidth=1.2, label="In progress")
    style_axes(ax, "Time (s)", "Subtasks (smoothed)")
    ax.legend(fontsize=8 * _FONT_SCALE)
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f"sim{idx}_required_vs_executing.pdf"), bbox_inches="tight")
    plt.show()

## 6. Cumulative initiated requests, per request type

In [ ]:
def plot_cumulative_requests(trajectory, timesteps, idx):
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    for t in [2, 0, 3, 1]:
        ax.plot(timesteps, trajectory.cum_active_queries[t],
                color=_theme_colors[t % len(_theme_colors)],
                linestyle=_REQUEST_LINESTYLES[t % len(_REQUEST_LINESTYLES)],
                linewidth=1.2, label=REQUEST_TYPE_NAMES[t])
    style_axes(ax, "Time (s)", "Cumulative requests")
    ax.legend(fontsize=8 * _FONT_SCALE)
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, f"sim{idx}_cumulative_requests.pdf"), bbox_inches="tight")
    plt.show()

## Generate plots for every experiment

In [ ]:
PLOTS = [
    plot_online_offline_users,
    plot_active_requests,
    plot_cpu_usage,
    plot_stored_partitions,
    plot_required_vs_executing,
    plot_cumulative_requests,
]

for idx, path, exp in records:
    print(f"=== {describe(idx, exp)} ===")
    with open(path, "rb") as f:
        trajectory: TrajectoryWithMetrics = CPU_Unpickler(f).load()
    timesteps = np.arange(len(trajectory.n_online_clients))

    for plot in PLOTS:
        plot(trajectory, timesteps, idx)

    del trajectory